# 위하고 엑셀 파일 테스트 

# 근로소득

In [2]:
from pathlib import Path
import pandas as pd

# 엑셀 파일 경로 (노트북 기준 현재 작업 폴더에서 탐색)
file_name = "종로구육아종합지원센터-202512.xlsx"
file_path = Path(file_name)

if not file_path.exists():
    # 워크스페이스 전체에서 동일 파일명 검색
    candidates = list(Path('.').rglob(file_name))
    if candidates:
        file_path = candidates[0]
        print(f"파일을 찾았습니다: {file_path}")
    else:
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {file_name}")


def _clean(v):
    if pd.isna(v):
        return ""
    text = str(v).strip()
    if text.lower().startswith("unnamed"):
        return ""
    return text


def build_columns_from_two_header_rows(sheet_df):
    h1 = sheet_df.iloc[0].tolist()  # 1행 헤더
    h2 = sheet_df.iloc[1].tolist()  # 2행 헤더

    columns = []
    for a, b in zip(h1, h2):
        a = _clean(a)
        b = _clean(b)

        if a and b and a != b:
            col = f"{a}_{b}"   # 예: 수당_기본급
        elif b:
            col = b              # 2행 우선
        elif a:
            col = a              # 2행 비면 1행 사용
        else:
            col = ""

        columns.append(col)

    # 빈 컬럼명/중복 컬럼명 정리
    seen = {}
    normalized = []
    for idx, col in enumerate(columns, start=1):
        if not col:
            col = f"col_{idx}"

        seen[col] = seen.get(col, 0) + 1
        if seen[col] > 1:
            col = f"{col}_{seen[col]}"

        normalized.append(col)

    return normalized


# 1) 시트 목록 확인
excel_file = pd.ExcelFile(file_path)
print("시트 목록:", excel_file.sheet_names)

# 2) 첫 번째 시트: 헤더 2행 병합 후 데이터 읽기
first_sheet = excel_file.sheet_names[0]
raw_first = pd.read_excel(file_path, sheet_name=first_sheet, header=None)
columns_first = build_columns_from_two_header_rows(raw_first)

df = raw_first.iloc[2:].copy()     # 데이터는 3행부터
df.columns = columns_first
df = df.reset_index(drop=True)

print(f"\n[첫 시트: {first_sheet}] 행/열 크기:", df.shape)
print("컬럼:", df.columns.tolist())
display(df.head())

# 3) 모든 시트 상위 3행 미리보기 (헤더 2행 병합 적용)
for sheet in excel_file.sheet_names:
    raw = pd.read_excel(file_path, sheet_name=sheet, header=None)
    cols = build_columns_from_two_header_rows(raw)
    preview = raw.iloc[2:].copy()
    preview.columns = cols
    preview = preview.reset_index(drop=True).head(3)

    print(f"\n===== 시트: {sheet} =====")
    display(preview)

파일을 찾았습니다: wehago/종로구육아종합지원센터-202512.xlsx
시트 목록: ['Sheet1']

[첫 시트: Sheet1] 행/열 크기: (31, 29)
컬럼: ['사원코드', '사원명', '부서', '직급', '직종', '수당_기본급', '상여', '식대', '수당', '출장비', '지급액계', '공제_국민연금', '건강보험', '고용보험', '장기요양보험료', '소득세', '지방소득세', '학자금상환액', '건강보험정산', '국민연금정산', '고용보험정산', '1-3월 건강보험미납분', '장기요양정산', '기납부소득세', '기납부지방소득세', '추가정산소득세', '추가정산지방소득세', '공제액계', '차인지급액']


/Users/yong/Documents/admin-system-saas/.venv_excel/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,사원코드,사원명,부서,직급,직종,수당_기본급,상여,식대,수당,출장비,...,국민연금정산,고용보험정산,1-3월 건강보험미납분,장기요양정산,기납부소득세,기납부지방소득세,추가정산소득세,추가정산지방소득세,공제액계,차인지급액
0,1,류외희,NaN,직급없음,NaN,5277560,0,150000,2584980,0,...,0,0,0,0,0,0,0,0,1341330,6671210
1,10,이신정,NaN,직급없음,NaN,2958300,0,0,500000,0,...,0,0,0,0,0,0,0,0,406790,3051510
2,19,엄지홍,NaN,직급없음,NaN,2714400,0,150000,1161560,0,...,0,0,0,0,0,0,0,0,557350,3468610
3,47,배은수,NaN,직급없음,NaN,2123000,0,140000,791100,0,...,0,0,0,0,0,0,0,0,334710,2719390
4,56,노수연,NaN,직급없음,NaN,1000000,0,75000,442500,0,...,0,0,0,0,0,0,0,0,136830,1380670



===== 시트: Sheet1 =====


/Users/yong/Documents/admin-system-saas/.venv_excel/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,사원코드,사원명,부서,직급,직종,수당_기본급,상여,식대,수당,출장비,...,국민연금정산,고용보험정산,1-3월 건강보험미납분,장기요양정산,기납부소득세,기납부지방소득세,추가정산소득세,추가정산지방소득세,공제액계,차인지급액
0,1,류외희,NaN,직급없음,NaN,5277560,0,150000,2584980,0,...,0,0,0,0,0,0,0,0,1341330,6671210
1,10,이신정,NaN,직급없음,NaN,2958300,0,0,500000,0,...,0,0,0,0,0,0,0,0,406790,3051510
2,19,엄지홍,NaN,직급없음,NaN,2714400,0,150000,1161560,0,...,0,0,0,0,0,0,0,0,557350,3468610


## 사업 소득

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

# ── 파일 경로 ──
swbu_file_name = "SWBU0102_2021_03.xls"
swbu_file_path = Path(swbu_file_name)

if not swbu_file_path.exists():
    candidates = list(Path('.').rglob(swbu_file_name))
    if candidates:
        swbu_file_path = candidates[0]
        print(f"파일을 찾았습니다: {swbu_file_path}")
    else:
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {swbu_file_name}")

raw = pd.read_excel(swbu_file_path, engine="xlrd", header=None)

# ── 메타 정보(제목, 회사명) 추출 ──
title = str(raw.iloc[0, 5]).strip()       # (2025년12월) 사업소득지급대장(합계)
company = str(raw.iloc[2, 0]).strip()     # 회사명:서울유캔수학학원
print(f"제목: {title}")
print(f"{company}\n")

# ── 2행 헤더 병합 (Row 4-5) ──
h1 = raw.iloc[4].tolist()   # 1행 헤더
h2 = raw.iloc[5].tolist()   # 2행 헤더

def _clean(v):
    if pd.isna(v):
        return ""
    return str(v).strip()

# 1행만 있는 컬럼 / 2행만 있는 컬럼 / 둘 다 있는 컬럼 처리
col_row1 = []  # 1행 헤더 기준 컬럼명
col_row2 = []  # 2행 헤더 기준 컬럼명

for a, b in zip(h1, h2):
    a, b = _clean(a), _clean(b)
    col_row1.append(a if a else None)
    col_row2.append(b if b else None)

print("1행 헤더:", col_row1)
print("2행 헤더:", col_row2)

# ── 데이터 영역 (Row 6~) 을 2행씩 묶어 1인 1행으로 변환 ──
data_rows = raw.iloc[6:].reset_index(drop=True)
records = []

for i in range(0, len(data_rows), 2):
    row1 = data_rows.iloc[i]
    row2 = data_rows.iloc[i + 1] if i + 1 < len(data_rows) else pd.Series([None] * len(row1))

    record = {}
    for col_idx in range(len(col_row1)):
        name1 = col_row1[col_idx]
        name2 = col_row2[col_idx]
        val1 = row1.iloc[col_idx] if not pd.isna(row1.iloc[col_idx]) else None
        val2 = row2.iloc[col_idx] if not pd.isna(row2.iloc[col_idx]) else None

        if name1 and name2:
            # 두 행 모두 헤더가 있는 컬럼 → 각각 별도 컬럼
            record[name1] = val1
            record[name2] = val2
        elif name1:
            # 1행에만 헤더 → 1행 데이터만 사용
            record[name1] = val1
        elif name2:
            # 2행에만 헤더 → 2행 데이터만 사용
            record[name2] = val2

    records.append(record)

df_swbu = pd.DataFrame(records)

print(f"\n변환 완료: {df_swbu.shape[0]}행 × {df_swbu.shape[1]}열")
print("컬럼:", df_swbu.columns.tolist())
display(df_swbu)

파일을 찾았습니다: wehago/SWBU0102_2021_03.xls
제목: (2025년12월) 사업소득지급대장(합계)
회사명:서울유캔수학학원

1행 헤더: ['NO', '코드', '성   명', '귀속년월', '지급액', '소득세', '예술/특고인 경비', '학자금상환액', None, '영수인']
2행 헤더: [None, None, None, '지급년월', None, '지방소득세', '고용보험료', '산재보험료', '차인지급액', None]

변환 완료: 5행 × 14열
컬럼: ['NO', '코드', '성   명', '귀속년월', '지급년월', '지급액', '소득세', '지방소득세', '예술/특고인 경비', '고용보험료', '학자금상환액', '산재보험료', '차인지급액', '영수인']


,NO,코드,성 명,귀속년월,지급년월,지급액,소득세,지방소득세,예술/특고인 경비,고용보험료,학자금상환액,산재보험료,차인지급액,영수인
0,1,000006,김선례,2025.12,2025.12,7000000,210000,21000,None,None,None,None,6769000,None
1,2,000007,전영수,2025.12,2025.12,7500000,225000,22500,None,None,None,None,7252500,None
2,3,000019,김선주,2025.12,2025.12,300000,9000,900,None,None,None,None,290100,None
3,4,000020,박혜영,2025.12,2025.12,1350000,40500,4050,None,None,None,None,1305450,None
4,총 계,NaN,NaN,NaN,NaN,16150000,484500,48450,None,None,None,None,15617050,None


## 기타 소득

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

# ── 파일 경로 ──
swet_file_name = "SWET0102_01_2010.xls"
swet_file_path = Path(swet_file_name)

if not swet_file_path.exists():
    candidates = list(Path('.').rglob(swet_file_name))
    if candidates:
        swet_file_path = candidates[0]
        print(f"파일을 찾았습니다: {swet_file_path}")
    else:
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {swet_file_name}")

raw = pd.read_excel(swet_file_path, engine="xlrd", header=None)

# ── 메타 정보 추출 ──
title = str(raw.iloc[0, 5]).strip()       # (2025.12) 기타소득 지급 대장(합계)
period = str(raw.iloc[0, 3]).strip()      # (2025.12)
company = str(raw.iloc[2, 0]).strip()     # 종로구육아종합지원센터
print(f"제목: {title}")
print(f"기간: {period}")
print(f"회사명: {company}\n")

# ── 2행 헤더 병합 (Row 3-4) ──
h1 = raw.iloc[3].tolist()   # 1행 헤더: NO, 코드, 성명, 소득구분, 귀속년월, 지급액, 소득세/법인세, 차인지급액, 영수인
h2 = raw.iloc[4].tolist()   # 2행 헤더: 지급년월, 지방소득세

def _clean(v):
    if pd.isna(v):
        return ""
    return str(v).strip()

col_row1 = []
col_row2 = []

for a, b in zip(h1, h2):
    a, b = _clean(a), _clean(b)
    col_row1.append(a if a else None)
    col_row2.append(b if b else None)

print("1행 헤더:", col_row1)
print("2행 헤더:", col_row2)

# ── 데이터 영역 (Row 5~) 을 2행씩 묶어 1인 1행으로 변환 ──
data_rows = raw.iloc[5:].reset_index(drop=True)
records = []

for i in range(0, len(data_rows), 2):
    row1 = data_rows.iloc[i]
    row2 = data_rows.iloc[i + 1] if i + 1 < len(data_rows) else pd.Series([None] * len(row1))

    record = {}
    for col_idx in range(len(col_row1)):
        name1 = col_row1[col_idx]
        name2 = col_row2[col_idx]
        val1 = row1.iloc[col_idx] if not pd.isna(row1.iloc[col_idx]) else None
        val2 = row2.iloc[col_idx] if not pd.isna(row2.iloc[col_idx]) else None

        if name1 and name2:
            record[name1] = val1
            record[name2] = val2
        elif name1:
            record[name1] = val1
        elif name2:
            record[name2] = val2

    records.append(record)

df_swet = pd.DataFrame(records)

print(f"\n변환 완료: {df_swet.shape[0]}행 × {df_swet.shape[1]}열")
print("컬럼:", df_swet.columns.tolist())
display(df_swet)

파일을 찾았습니다: wehago/SWET0102_01_2010.xls
제목: 기타소득 지급 대장(합계)
기간: (2025.12)
회사명: 종로구육아종합지원센터

1행 헤더: ['NO', '코드', '성   명', '소득구분', '귀속년월', '지급액', '소득세/법인세', '차인지급액', '영수인']
2행 헤더: [None, None, None, None, '지급년월', None, '지방소득세', None, None]

변환 완료: 4행 × 11열
컬럼: ['NO', '코드', '성   명', '소득구분', '귀속년월', '지급년월', '지급액', '소득세/법인세', '지방소득세', '차인지급액', '영수인']


,NO,코드,성 명,소득구분,귀속년월,지급년월,지급액,소득세/법인세,지방소득세,차인지급액,영수인
0,1,000160,김성화,강연료 등,2025.12,2025.12.25,120000,0,0,120000,None
1,2,000206,한영미,강연료 등,2025.12,2025.12.25,275000,22000,2200,250800,None
2,3,000153,유도연,강연료 등,2025.12,2025.12.25,300000,24000,2400,273600,None
3,총 합 계,NaN,NaN,NaN,NaN,NaN,695000,46000,4600,644400,None


## 일용직 소득

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np

# ── 파일 경로 ──
swsa_file_name = "SWSA0107_2.xls"
swsa_file_path = Path(swsa_file_name)

if not swsa_file_path.exists():
    candidates = list(Path('.').rglob(swsa_file_name))
    if candidates:
        swsa_file_path = candidates[0]
        print(f"파일을 찾았습니다: {swsa_file_path}")
    else:
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {swsa_file_name}")

raw = pd.read_excel(swsa_file_path, engine="xlrd", header=None)

# ── 메타 정보 추출 ──
title = str(raw.iloc[0, 15]).strip()              # 일용직 급여 지급명세서
period_belong = str(raw.iloc[2, 10]).strip()       # 귀속: 2024년03월
period_pay = str(raw.iloc[2, 18]).strip()          # 지급: 2024년03월
company = str(raw.iloc[4, 1]).strip()              # 컴포즈커피 수원연무점
report_period = str(raw.iloc[4, 7]).strip()        # 신고: 2024년04월

print(f"제목: {title}")
print(f"귀속: {period_belong} / 지급: {period_pay}")
print(f"회사명: {company} / 신고: {report_period}\n")

# ── 2행 헤더 병합 (Row 6-7) ──
h1 = raw.iloc[6].tolist()
h2 = raw.iloc[7].tolist()

def _clean(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    return str(v).strip()

col_row1 = []
col_row2 = []

for a, b in zip(h1, h2):
    a, b = _clean(a), _clean(b)
    col_row1.append(a if a else None)
    col_row2.append(b if b else None)

print("1행 헤더:", col_row1)
print("2행 헤더:", col_row2)

# ── 데이터 영역 (Row 8~) 을 2행씩 묶어 1인 1행으로 변환 ──
data_rows = raw.iloc[8:].reset_index(drop=True)
records = []

for i in range(0, len(data_rows), 2):
    row1 = data_rows.iloc[i]
    row2 = data_rows.iloc[i + 1] if i + 1 < len(data_rows) else pd.Series([None] * len(row1))

    record = {}
    for col_idx in range(len(col_row1)):
        name1 = col_row1[col_idx]
        name2 = col_row2[col_idx]

        def _val(row, idx):
            v = row.iloc[idx]
            if v is None:
                return None
            if isinstance(v, float) and np.isnan(v):
                return None
            return v

        val1 = _val(row1, col_idx)
        val2 = _val(row2, col_idx)

        if name1 and name2:
            record[name1] = val1
            record[name2] = val2
        elif name1:
            record[name1] = val1
        elif name2:
            record[name2] = val2

    records.append(record)

df_swsa = pd.DataFrame(records)

print(f"\n변환 완료: {df_swsa.shape[0]}행 × {df_swsa.shape[1]}열")
print("컬럼:", df_swsa.columns.tolist())
display(df_swsa)

파일을 찾았습니다: wehago/SWSA0107_2.xls
제목: 일용직 급여 지급명세서
귀속: 2024년03월 / 지급: 2024년03월
회사명: 컴포즈커피 수원연무점 / 신고: 2024년04월

1행 헤더: ['코드', '직종', '주민등록번호', '전화번호', '입사일', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', None, '근무', '근무시간', None, '급여총액', '국민연금', '소득세', '요양보험', '영수인']
2행 헤더: ['성명', '직급', '주 소', None, None, '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31.0', '일수', '정상', '연장', '고용보험료', '건강보험료', '지방소득세', '차인지급액', None]

변환 완료: 5행 × 53열
컬럼: ['코드', '성명', '직종', '직급', '주민등록번호', '주 소', '전화번호', '입사일', '1', '16', '2', '17', '3', '18', '4', '19', '5', '20', '6', '21', '7', '22', '8', '23', '9', '24', '10', '25', '11', '26', '12', '27', '13', '28', '14', '29', '15', '30', '31.0', '근무', '일수', '근무시간', '정상', '연장', '급여총액', '고용보험료', '국민연금', '건강보험료', '소득세', '지방소득세', '요양보험', '차인지급액', '영수인']


,코드,성명,직종,직급,주민등록번호,주 소,전화번호,입사일,1,16,...,연장,급여총액,고용보험료,국민연금,건강보험료,소득세,지방소득세,요양보험,차인지급액,영수인
0,5,국지수,None,None,040727-*******,,None,2022.07.25,NaN,None,...,None,880000,7910,None,None,None,None,None,872090,None
1,9,김민정,None,None,000511-*******,,None,2022.08.31,NaN,None,...,None,876750,7850,None,None,None,None,None,868900,None
2,13,이다현,None,None,030222-*******,,None,2023.03.04,NaN,None,...,None,625000,5620,None,None,None,None,None,619380,None
3,14,변승민,None,None,010517-*******,,None,2023.05.01,NaN,None,...,None,758333,6810,None,None,None,None,None,751523,None
4,총인원,4명,None,None,NaN,NaN,None,NaN,합 계,None,...,None,3140083,28190,None,None,None,None,None,3111893,None
